<a href="https://colab.research.google.com/github/YKTTKY/Book-BuildALargeLanguageModel-FromScratch-/blob/main/BuildLLMFromScratch_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Attention Mechanism
compute attention weights by relating different positions within a single input sequence.  


In [ ]:
# simplified attention for conceptual understanding
import torch
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]  # step     (x^6)
])

In [ ]:
# compute the context vector, which create enriched representations of each element in an input sequence
# step 1 : compute the attention scores about x^2
query = inputs[1] # (x^2)
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


the dot product is a measure of similarity because it quantifies how closely two vectors are aligned:
```
attn_scores_2[i] = torch.dot(x_i, query)
```

In [ ]:
# step 2 : normalize to obtain attention weights that sums to 1.
# purpose : maintain training stability in an LLM.
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights: ", attn_weights_2_tmp)
print("Sum: ", attn_weights_2_tmp.sum())
print(attn_scores_2.sum())

Attention weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum:  tensor(1.0000)
tensor(6.5617)


### softmax
<img src="https://miro.medium.com/1*ybZLF38Q-PmEz2aJZJPo8A.png" alt="softmax function" width=400>

In [ ]:
# Softmax Normalization:
# 1. always positive value
# 2. interpretable as probabilities
# 3. relative importance
def softmax_naive(x):
  return torch.exp(x)/ torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention Weights: ", attn_weights_2_naive)
print("Sum: ", attn_weights_2_naive.sum())

Attention Weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


In [ ]:
# PyTorch Softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention Weights: ", attn_weights_2)
print("Sum: ", attn_weights_2.sum())

Attention Weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


In [ ]:
# step 3 : calculate the context vector:
query = inputs[1] # (x^2)
context_vec_2  = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Computing attention weights for all input tokens
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-11.png" alt="attention pattern image.">

In [ ]:
# compute the entire attention scores:
# input -> Your journey starts with one step ( 6 words )

attn_scores = torch.empty(6, 6)
# step 1: compute the dot products between inputs
for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
# better way to perform dot products (fast computation):
# @ -> matrix multiplications which is calling torch.matmul(a,b)
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
# step 2: normalize the attn_scores with softmax
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In the context of PyTorch, the `dim` parameter in functions like `torch.softmax` specifies the dimensions of the input tensor along which the function will be computed. By setting `dim=-1`, we are instructing the `softmax` function to apply normalization along the `last dimension` of the `attn_scores` tensor.   
In this example, the `attn_scores` is a two-dimensional tensor with a shape of [6,6], it will <mark>normalize across the columns</mark> so that the values in <mark>each row (summing over the column dimension) sum up to 1</mark>.

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 Sum: ", row_2_sum)
print("All Row Sums: ", attn_weights.sum(dim=-1))

Row 2 Sum:  1.0
All Row Sums:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [ ]:
# step 3: attention weight(attention pattern) to compute all context vectors.
all_context_vec = attn_weights @ inputs
print(all_context_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [ ]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


## Implementating self-attention with trainable weights.
Tranable weights are built on the previous concepts: we want to compute context vectors as weighted sums over the input vectors specific to a certain element.  
The most notable difference is the introduciton weight matrices that are updated during model training. These trainable weights matrices are crucial to produce "good" context vectors.
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-14.png" alt="qkv mechanism">  
Three Trainable weight matrices $W_q$, $W_k$, and $W_v$ are used to project the embedded input tokens, $x^i$

In [ ]:
# computing one context vector x^(2):
x_2 = inputs[1] # x^(2)
d_in = inputs.shape[1] # input embedding size -> d = 3
d_out = 2 # output embedding size -> d_out = 2

In [ ]:
# initialize three matices W_q, W_k, W_v
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


### WEIGHT PARAMETERS VS. ATTENTION WEIGHTS  
`weight` is refer to the values of a neural network.  
- $W_q$, $W_k$, $W_v$

`attention weight` is refer to the extent to which a context vector depends on different parts of input.  
- results of $QK^T$.

Comparision
|Feature| Weight parameters(W) | Attention Weights(A) |
| -------- | --------| --------|
|Origin| Learned from billions of words.| Calculated from specific prompt.|
|Stability| Static( Does not change during use | Dynamic ( Changes with every word).|
|Storage| Stored in the model(Gbs)| Exist only in RAM for a millisecond|
Purpose| Defines the "rules" of the world| Define the "context" of a sentence.



project the six input tokens from a three-dimensional onto a two-dimensional embedding space:

In [ ]:
# obtain kets and values :
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape: ", keys.shape)
print("values.shape", values.shape)

keys.shape:  torch.Size([6, 2])
values.shape torch.Size([6, 2])


In [ ]:
# compute the attention score for x^2
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2) # Q K^T
print(attn_score_22)

tensor(1.8524)


In [ ]:
# generalize computation to all attention scores : x^2 respect to other inputs
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [ ]:
# from attention scores to attention weights by scaling the scores and softmax function
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim = -1 )
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### **Note : why scaled-dot product attention**
Improve the training performance by avoiding small gradients.  
Large  dot products can result in very small gradients during backpropogation due to the softmax function applied to them. As the dot product increase, the softmax function behaves more like a step function, resuling in gradients nearing zero and hence slow down the training process in stagnation.

In [ ]:
# compute the context vector:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### **Implementating a compact self-attention Python class**  

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x):
    # obtain Q, K, V matrices from weights W_q, W_k, and W_v:
    keys = x @ self.W_key # K
    queries = x @ self.W_query # Q
    values = x @ self.W_value # V
    # calculate attention scores (similarity)
    attn_scores = queries @ keys.T
    # nomralize with softmax and d_k (embedding dimension)
    attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim = -1)
    # return context vector
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
# self attention class using PyTorch's Linear Layers:
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax( attn_scores / keys.shape[-1] ** 0.5, dim=-1)
    context_vec = attn_weights @ values
    return context_vec


In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [ ]:
print(sa_v1.W_query.shape)
print(sa_v2.W_query.weight.T.shape)

torch.Size([3, 2])
torch.Size([3, 2])


In [ ]:
# transfer the v2 weight to v1 :
sa_v1.W_query = torch.nn.Parameter(torch.transpose(sa_v2.W_query.weight, 0, 1))
sa_v1.W_key = torch.nn.Parameter(torch.transpose(sa_v2.W_key.weight, 0, 1))
sa_v1.W_value = torch.nn.Parameter(torch.transpose(sa_v2.W_value.weight, 0, 1) )
print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Applying a Casual Attention Mask
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-20.png" alt="steps to casual mask">

In [ ]:
# step 1 : compute the attention weights
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
# step 2: create a mask with Pytorch's 'tril' function
# values above the diagonal are zero
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
# multiply the attention weights to zero-out the values above the diagonal:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [ ]:
# step 3: normalize the attention weights to sum up to 1 again in each row:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


### INFORMATION LEAKAGE:  
zero-out attention weights in the previous steps do not achieve what we want.  What we want is to really hide the future token values. Putting zero in the softmax result the $e^0$ = 1, therefore, we set the value to negative infinity to not contribute to the softmax value .  

$e^{-\infty} = 0$

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim = -1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


We could use the modified attention weights to compute the context vectors via `context_vec = attn_weights @ values`. However, we will first cover another minar tweak to the casual attention mechanism that is useful for reducing overfitting when training LLMs.

### **Masking additional attention weights with dropout**  
Disgard randomly hidden layer in the neural network to avoid overfitting issues.  
In transformer architectures, there are two specific times:  
1. after calculating the attention weight
2. after applying the attention weights to the value vectors.
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-22.png" alt="drop out">


In [ ]:
# approximately half of the values are zeroed out:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%.
example = torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
## apply the dropout the attention weights:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
# implementating a compact casual attention class
batch = torch.stack((inputs, inputs), dim=0) # two inputs with six tokens each; each tokens has embedding dimensions3.
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
# casual attention class
class CasualAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key  = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout) #1 Compared to the previous SelfAttention_v1 class, we added a dropout layer.
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    ) #2 The register_buffer call is also a new addition (more information is provided in the following text).
  def forward(self, x):
    b, num_tokens, d_in = x.shape #3 We transpose dimensions 1 and 2, keeping the batch dimension at the first position (0).
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
    )#4 In PyTorch, operations with a trailing underscore are performed in-place, avoiding unnecessary memory copies.
    attn_weights = torch.softmax( attn_scores / keys.shape[-1] ** 0.5, dim=-1)
    attn_weights - self.dropout(attn_weights)
    context_vec = attn_weights @ values
    return context_vec



In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CasualAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(context_vecs)
print("context_vecs.shape: ", context_vecs.shape)

tensor([[[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]],

        [[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape:  torch.Size([2, 6, 2])


## **Extending Single-head attention to multi-head attention**  
`Multihead attention` refers to dividing the attention mechanism into multiple `heads`, each operating independently.  
To do this, we can 'stack' up multiple Casual Attention modules.  
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-24.png" alt="multi head attention">


In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, nums_heads, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
        [ CasualAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(nums_heads) ]
    )
  def forward(self, x):
    return torch.cat( [head(x) for head in self.heads], dim= -1 )

<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/3-25.png" alt="dimension of output context vectors">  
 Using the MultiHeadAttentionWrapper, we specified the number of attention heads (num_heads). If we set num_heads=2, as in this example, we obtain a tensor with two sets of context vector matrices. In each context vector matrix, the rows represent the context vectors corresponding to the tokens, and the columns correspond to the embedding dimension specified via d_out=4. We concatenate these context vector matrices along the column dimension. Since we have two attention heads and an embedding dimension of 2, the final embedding dimension is 2 × 2 = 4.

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] # number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in,
                                d_out,
                                context_length,
                                0.0,
                                nums_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print(f"d_in: {d_in} , d_out: {d_out}")
print("context_vecs.shape: ", context_vecs.shape)

tensor([[[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
d_in: 3 , d_out: 2
context_vecs.shape:  torch.Size([2, 6, 4])


In [ ]:
# change the input arguments for MultiHeadAttentionWrapper while keeping num_heads = 2 to make context_vecs two-dimensional instead of four.
torch.manual_seed(123)
context_length = batch.shape[1] # number of tokens
d_in, d_out = 3, 1
mha = MultiHeadAttentionWrapper(d_in,
                                d_out,
                                context_length,
                                0.0,
                                nums_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print(f"d_in: {d_in} , d_out: {d_out}")
print("context_vecs.shape: ", context_vecs.shape)

tensor([[[-0.6432, -0.1042],
         [-0.6430, -0.1054],
         [-0.6430, -0.1054],
         [-0.6423, -0.1061],
         [-0.6426, -0.1049],
         [-0.6424, -0.1065]],

        [[-0.6432, -0.1042],
         [-0.6430, -0.1054],
         [-0.6430, -0.1054],
         [-0.6423, -0.1061],
         [-0.6426, -0.1049],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
d_in: 3 , d_out: 1
context_vecs.shape:  torch.Size([2, 6, 2])


## **Implementing multihead attention with weight splits**  


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads    #1 Reduces the projection dim to match the desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)    #2 Uses a Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)

        """
        If you just assigned self.mask = torch.triu(...), and then called model.to("cuda"),
        your model weights would move to the GPU,
        but your mask would stay on the CPU, causing a crash. register_buffer prevents this.
        """
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)         #3 Tensor shape: (b, num_tokens, d_out)
        queries = self.W_query(x)    #3 Tensor shape: (b, num_tokens, d_out)
        values = self.W_value(x)     #3 Tensor shape: (b, num_tokens, d_out)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)       #4 We implicitly split the matrix by adding a num_heads dimension. Then we unroll the last dim: (b, num_tokens, d_out) -&gt; (b, num_tokens, num_heads, head_dim).
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )

        keys = keys.transpose(1, 2)          #5 Transposes from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2)    #5 Transposes from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)
        values = values.transpose(1, 2)      #5 Transposes from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)
        # Q( T x d_k ) K( T x dk ) transpose(2,3) -> KT( dk x T )
        attn_scores = queries @ keys.transpose(2, 3)   #6 Computes dot product for each head
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]    #7  Masks truncated to the number of tokens

        attn_scores.masked_fill_(mask_bool, -torch.inf)     #8 Uses the mask to fill attention scores

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)   #9 Tensor shape: (b, num_tokens, n_heads, head_dim)
 #10 Combines heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)    #11 Adds an optional linear projection
        return context_vec

In [ ]:
torch.manual_seed(123)
batch_size, context_length,d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print("batch_size:", batch_size)
print("context_length:", context_length)
print("d_in:", d_in )
print("d_out:", d_out)
print(context_vecs)
print("context_vecs.shape: ", context_vecs.shape)

batch_size: 2
context_length: 6
d_in: 3
d_out: 2
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape:  torch.Size([2, 6, 2])


In [ ]:
## Initialize GPT-2 size attention modules
gpt2_context_length = 1024
gpt2_d_in, gpt_2_d_out = (768, 768)
gpt2_num_heads = 12
gpt2 = MultiHeadAttention(gpt2_d_in, gpt_2_d_out, gpt2_context_length, 0.0, num_heads=gpt2_num_heads)

In [ ]:
batch_size = 1
num_tokens = 3 # "I", "Love", "You"
# shape : batch, tokens, d_in
x = torch.randn(batch_size, num_tokens, gpt2_d_in)
out = gpt2(x)
print(f"input shape: {x.shape}")
print(f"output shape: {out.shape}")


input shape: torch.Size([1, 3, 768])
output shape: torch.Size([1, 3, 768])
